# 8.8 保留原權重，能只學一條小修正嗎？


要教模型一種新寫法，不一定要更新每個原有數字。先看一個線性層：16個輸入乘權重矩陣W，變成12個輸出。W有`16×12=192`個數字。希望保留這條原路，同時加上可訓練的小修正。

**LoRA**（Low-Rank Adaptation）讓修正先經矩陣A：16個數變成2個，再經B：2個變成12個，最後加回原輸出。中間寬度2叫rank；A、B共`2×16+12×2=56`個數字。原W凍結，表示訓練不更新它；A、B可以學習。修正先通過窄通道，能表達的矩陣變化受到限制，這就是此處「低秩」的意義。

公式寫成`ΔW=(alpha/rank)BA`，ΔW是加到W的改變量，alpha是縮放設定。本例alpha與rank都是2，倍率為1。圖中的兩條路使用同一份輸入，在相加處才合流。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-08-lora-paths.svg")))

In [ ]:
import torch
from torch import nn
from tiny_perceptron.alignment import LoRALinear

torch.manual_seed(0)
layer = LoRALinear(nn.Linear(16, 12), rank=2, alpha=2)
x = torch.randn(3, 16)
print("可訓練參數", sum(p.numel() for p in layer.parameters() if p.requires_grad))
print("初始相同", torch.equal(layer(x), layer.base(x)))
layer(x).sum().backward()
print("A梯度為零", layer.a.grad.abs().max().item() == 0)
print("B梯度非零", layer.b.grad.abs().max().item() > 0)

`numel()`數元素，`requires_grad`篩出可學部分，因此可訓練參數為56。初始輸出與原層相同：B初始全零，修正為零。`sum().backward()`只計算輸出加總的梯度，沒有更新器的`step()`，也沒有風格示範，因此不是模型學會新寫法的證據。

後兩行都是`True`：首次A梯度全零、B已有非零梯度。A的影響須經仍為零的B，才會出現這個初始現象；B更新後，A才可能收到訊號。原層另有12個偏置，也被凍結，192只指W。

真正教風格時，把修正插到完整模型，再用回答目標更新A、B，並與完整微調比較。同一保存實驗中，固定積木句的LoRA版本在七題新加法有6題符合指定句，完整微調有7題；兩者都沒有答對新加法。少更新參數是確定的結構收益，寫法與內容效果仍由測試決定。原權重、運算和中間資料仍存在，56個可學數字也不是整個模型只佔56個數字。

練習把rank與alpha都改成4，維持倍率1。先算可訓練數字為`4×(16+12)=112`，再核對輸出；窄通道變寬，同時增加要學的參數。

<details>
<summary>補充：既有實驗的條件與完整紀錄</summary>

把這條補充路徑放進完整模型後，我們也真的訓練了兩套adapter，也就是各自為某種寫法訓練的一組A、B修正參數。起點是[8.3的加法基模](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)，最後七道新題原本就零題答對。兩套各讀相同的49道訓練題，一套學短答、一套學固定積木句，各更新450次；模型兩層的Query、Value、注意力輸出、前饋兩個線性層，以及最後輸出層，共11處加上rank 4、alpha 4的修正。

這些插入位置都是像前面W一樣的線性變換。Query把當前內容轉成「想找什麼」的特徵，Value把各位置內容轉成「取回什麼」的特徵，分別可回看[3.3的查詢](https://birdhackor.github.io/tiny-perceptron-vlm/3.3.html)與[3.4的取回內容](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html)；注意力輸出再混合取回的結果。[4.4的前饋層](https://birdhackor.github.io/tiny-perceptron-vlm/4.4.html)對每個位置先擴展、再收回特徵，而[最後輸出層](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)把特徵轉成下一個文字片段的候選分數。每層有五處，兩層共十處，再加最後輸出層便是十一處。原本141,568個參數全部凍結，只更新9,504個新參數，約為原模型參數量的6.7%。第一步也出現上述A梯度全零、B已有非零梯度的現象，訓練前後的原模型數字則完全相同。

少更新數字，是否仍能學到寫法？我們再從同一起點完整更新全部參數，使用與比喻adapter相同的資料、450步和318,991個有效回答目標，作為對照。以下都看沒有放進訓練的題目；「風格符合」只檢查短答是否為數字，或回答是否含本次指定的積木句。

| 更新方式 | 可訓練參數 | 八題驗證的風格符合 | 七題最後檢查的風格符合 | 七題算術答對 |
| --- | ---: | ---: | ---: | ---: |
| LoRA短答 | 9,504 | 8／8 | 7／7 | 0／7 |
| LoRA固定比喻 | 9,504 | 5／8 | 6／7 | 0／7 |
| 全部參數學固定比喻 | 141,568 | 8／8 | 7／7 | 0／7 |

全部回答都有EOS，也就是回答結束標記，表示模型有生成停止訊號。能停下來，不保證答案正確或中間文字都能正常顯示；比喻adapter仍有一題生成`1�，像�兩組積木合在一起再數。`，其中`�`代替無法正常顯示的文字，這一題未通過固定句檢查。這次LoRA用較少更新參數學到了部分寫法，效果沒有完全追上完整微調；兩種方式也都沒有補上新加法的正確性。基模、前向計算與中間資料仍要留在記憶體，6.7%不等於整個訓練只花6.7%的記憶體，更不是模型總大小變成6.7%。[完整生成與比較報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/lora.json)保留這些失敗，不把較少參數直接換算成同等能力。

</details>
